In [1]:
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import re
import requests
%matplotlib inline

In [2]:
base_path ="/Users/emmagregory/Documents/Predicting London Property Prices/"
df_2022 = pd.read_csv(f"{base_path}propertysales2022.csv", low_memory=False)
df_2023 = pd.read_csv(f"{base_path}propertysales2023.csv", low_memory=False)

# Fix bad headers for 2024 and 2025
cols = ['Price', 'Date', 'Postcode', 'PropertyType', 'NewBuildFlag', 'Tenure',
        'Street', 'Town', 'District', 'County']

df_2024 = pd.read_csv(f"{base_path}propertysales2024.csv", header=None, names=cols, low_memory=False)
df_2025 = pd.read_csv(f"{base_path}propertysales2025.csv", header=None, names=cols, low_memory=False)


In [3]:
df_2025['PropertyType'] = df_2025['PropertyType'].fillna('Unknown')
df_2025['NewBuildFlag'] = df_2025['NewBuildFlag'].fillna('Unknown')
df_2025['Tenure'] = df_2025['Tenure'].fillna('Unknown')

In [4]:
def clean_df(df):
    df['Price'] = pd.to_numeric(df['Price'], errors='coerce')
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Postcode'] = df['Postcode'].astype(str).str.strip().str.upper()
    df['PropertyType'] = df['PropertyType'].astype(str).str.strip().str.upper()
    df['NewBuildFlag'] = df['NewBuildFlag'].astype(str).str.strip().str.upper()
    df['Tenure'] = df['Tenure'].astype(str).str.strip().str.upper()
    df['Street'] = df['Street'].astype(str).str.title().str.strip()
    df['Town'] = df['Town'].astype(str).str.title().str.strip()
    df['District'] = df['District'].astype(str).str.title().str.strip()
    df['County'] = df['County'].astype(str).str.title().str.strip()
    return df.dropna(subset=['Price', 'Date', 'Postcode'])

df_2022 = clean_df(df_2022)
df_2023 = clean_df(df_2023)
df_2024 = clean_df(df_2024)
df_2025 = clean_df(df_2025)

In [5]:
all_years_df = pd.concat([df_2022, df_2023, df_2024, df_2025], ignore_index=True)

In [6]:
unique_postcodes = all_years_df['Postcode'].unique()
print(f"Unique postcodes: {len(unique_postcodes)}")

Unique postcodes: 104142


In [7]:
def get_postcodes_batch(postcodes_list):
    """
    Fetch coordinates for up to 100 postcodes at once using batch API
    Returns a dictionary mapping postcode to (lat, lon)
    """
    url = "https://api.postcodes.io/postcodes"
    
    # API accepts max 100 postcodes per request
    batch = postcodes_list[:100]
    
    try:
        response = requests.post(url, json={"postcodes": batch})
        results = {}
        
        if response.status_code == 200:
            data = response.json()
            for item in data['result']:
                postcode = item['query']
                if item['result']:
                    results[postcode] = (item['result']['latitude'], item['result']['longitude'])
                else:
                    results[postcode] = (None, None)
        
        return results
    except Exception as e:
        print(f"Batch error: {e}")
        return {}

In [8]:
from tqdm import tqdm

# Create dictionary to store postcode -> (lat, lon) mapping
postcode_coords = {}

# Process in batches of 100
batch_size = 100
for i in tqdm(range(0, len(unique_postcodes), batch_size), desc="Processing batches"):
    batch = unique_postcodes[i:i+batch_size].tolist()
    batch_results = get_postcodes_batch(batch)
    postcode_coords.update(batch_results)
    
    # Small delay between batches
    time.sleep(0.1)

print(f"\nProcessed {len(postcode_coords)} unique postcodes")
print("Sample results:")
print(list(postcode_coords.items())[:5])

Processing batches: 100%|███████████████████| 1042/1042 [08:01<00:00,  2.16it/s]


Processed 104142 unique postcodes
Sample results:
[('W138PQ', (51.5155, -0.317544)), ('UB12BT', (51.520384, -0.376361)), ('UB25BY', (51.502532, -0.393163)), ('N97NL', (51.641781, -0.054572)), ('W72AT', (51.499959, -0.328058))]


In [9]:
# Map the coordinates back to the dataframe
all_years_df['lat'] = all_years_df['Postcode'].map(lambda x: postcode_coords.get(x, (None, None))[0])
all_years_df['long'] = all_years_df['Postcode'].map(lambda x: postcode_coords.get(x, (None, None))[1])

# Check the results
print("Preview of data with coordinates:")
print(all_years_df[['Postcode', 'lat', 'long']].head(10))

# Save to new CSV
all_years_df.to_csv('property_data_with_coordinates.csv', index=False)
print("\nSaved to: property_data_with_coordinates.csv")


Preview of data with coordinates:
  Postcode        lat      long
0   W138PQ  51.515500 -0.317544
1   UB12BT  51.520384 -0.376361
2   UB25BY  51.502532 -0.393163
3    N97NL  51.641781 -0.054572
4    W72AT  51.499959 -0.328058
5   W138PG  51.515768 -0.317419
6   HA72EP  51.603996 -0.313249
7   UB35HZ  51.481969 -0.422516
8    N31EB  51.605312 -0.197858
9  TW118SJ  51.431438 -0.342127

Saved to: property_data_with_coordinates.csv


In [10]:
# Check the shape and columns of what was saved
print(f"Shape: {all_years_df.shape}")
print(f"\nColumns: {all_years_df.columns.tolist()}")

Shape: (365427, 12)

Columns: ['Price', 'Date', 'Postcode', 'PropertyType', 'NewBuildFlag', 'Tenure', 'Street', 'Town', 'District', 'County', 'lat', 'long']


In [11]:
print(all_years_df.head)

<bound method NDFrame.head of             Price       Date Postcode PropertyType NewBuildFlag   Tenure  \
0       1110000.0 2022-06-17   W138PQ            T            N        F   
1        460000.0 2022-07-04   UB12BT            T            N        F   
2        425000.0 2022-06-28   UB25BY            T            N        F   
3        457000.0 2022-06-22    N97NL            T            N        F   
4        985000.0 2022-06-22    W72AT            S            N        F   
...           ...        ...      ...          ...          ...      ...   
365422   275000.0 2025-09-25  SW182ST      UNKNOWN      UNKNOWN  UNKNOWN   
365423   415000.0 2025-09-25  SW181UA      UNKNOWN      UNKNOWN  UNKNOWN   
365424   700000.0 2025-09-25  SW181GN      UNKNOWN      UNKNOWN  UNKNOWN   
365425   824000.0 2025-09-26  SW170SE      UNKNOWN      UNKNOWN  UNKNOWN   
365426   450000.0 2025-09-26  SW128DS      UNKNOWN      UNKNOWN  UNKNOWN   

                              Street      Town    Distric

In [12]:
save_path = "/Users/emmagregory/Documents/Predicting London Property Prices/property_data_with_coordinates.csv"
all_years_df.to_csv(save_path, index=False)
print(f"Saved to: {save_path}")

Saved to: /Users/emmagregory/Documents/Predicting London Property Prices/property_data_with_coordinates.csv
